[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/05_self_supervised/05_self_supervised.ipynb)

# 05 · 自监督视觉（纯 numpy 从零）

目标：把自监督两大范式从零写出来——**对比学习 InfoNCE/NT-Xent**（正对拉近、负对推开）与**掩码重建 MAE**（toy），再加 **表示坍缩诊断** 和 **linear probe 评估**，每步 `assert` 验证。

**路线**：
1. 增强对：正对(同图两视图) / 负对(不同图)
2. InfoNCE / NT-Xent 损失（= 在 2N−1 候选里认出正样本的交叉熵）
3. 温度 τ 的效应
4. MAE 掩码重建（toy，只在被掩块算 MSE）
5. 表示坍缩诊断 + linear probe 评估
6. ✏️ 练习 → 📖 答案 → 🧪 真实 optdigits 表示评估胶囊

> **本课纪律**：InfoNCE 必须排除自身、用余弦相似度(L2 归一)、正样本索引配对正确；用 `assert` 兜底。

## 1 · 增强对：正对与负对

对比学习的监督信号全来自成对关系：正对=同一图的两个增强视图(语义相同)，负对=不同图。
我们对一个 toy 向量数据集做随机增强(加噪+缩放)生成两视图，验证正对比负对更相似。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def augment_view(x, noise=0.1, seed=None):
    '''对样本做一次随机增强(加噪 + 轻微缩放), 模拟同图的一个视图。'''
    r = np.random.default_rng(seed)
    return x * (1 + 0.1*r.standard_normal(x.shape)) + noise*r.standard_normal(x.shape)

def l2norm(Z):
    return Z / (np.linalg.norm(Z, axis=-1, keepdims=True) + 1e-12)

# 4 个不同的 base 样本(语义)
base = rng.standard_normal((4, 8))
v1 = np.array([augment_view(base[i], seed=10+i) for i in range(4)])   # 视图1
v2 = np.array([augment_view(base[i], seed=20+i) for i in range(4)])   # 视图2
# 余弦相似度: 同图两视图(正对) 应高于 不同图(负对)
n1, n2 = l2norm(v1), l2norm(v2)
sim = n1 @ n2.T                                   # sim[i,j]=视图1的i 与 视图2的j
pos_sim = np.diag(sim).mean()                     # 正对(对角)
neg_sim = (sim.sum() - np.trace(sim)) / (16 - 4)  # 负对(非对角)
print(f'正对平均相似度={pos_sim:.3f}  负对平均相似度={neg_sim:.3f}')
assert pos_sim > neg_sim, '正对(同图)应比负对(不同图)更相似'
print('✅ 增强对构造正确：正对相似度 > 负对')

## 2 · InfoNCE / NT-Xent 损失

对每个视图 `i`，正样本是同图另一视图 `j(i)`，其余都是负样本。损失 = 把「认出正样本」当交叉熵：
`ℓ_i = -log( exp(sim(i,j)/τ) / Σ_{k≠i} exp(sim(i,k)/τ) )`。**排除自身**、用余弦相似度。

In [ ]:
def info_nce(z1, z2, tau=0.5):
    '''SimCLR NT-Xent: 2N 个视图(z1,z2 各 N), 每个的正样本是其孪生视图。
       返回平均损失。'''
    N = z1.shape[0]
    Z = l2norm(np.concatenate([z1, z2], axis=0))      # (2N, d), 余弦相似度先归一
    sim = Z @ Z.T / tau                               # (2N, 2N)
    np.fill_diagonal(sim, -1e9)                        # 排除自身(关键!)
    # 正样本索引: i 的正样本是 i+N (mod 2N)
    targets = np.concatenate([np.arange(N) + N, np.arange(N)])
    # 交叉熵: -log softmax(sim)[targets]
    sim = sim - sim.max(1, keepdims=True)
    logp = sim - np.log(np.exp(sim).sum(1, keepdims=True))
    return float(-logp[np.arange(2*N), targets].mean())

N = 4
# 好表示: 正对几乎相同, 负对随机 -> 损失低
z = rng.standard_normal((N, 16))
z1 = z + 0.01*rng.standard_normal((N, 16))
z2 = z + 0.01*rng.standard_normal((N, 16))
loss_good = info_nce(z1, z2, tau=0.5)
# 坏表示: 完全随机 -> 损失高(接近 log(2N-1))
loss_rand = info_nce(rng.standard_normal((N,16)), rng.standard_normal((N,16)), tau=0.5)
print(f'好表示 InfoNCE={loss_good:.3f}  随机表示 InfoNCE={loss_rand:.3f}  (上界≈log(2N-1)={np.log(2*N-1):.3f})')
assert loss_good < loss_rand, '正对相似的表示损失应更低'
assert loss_good >= 0, '损失非负'
print('✅ InfoNCE 正确：好表示损失低、随机表示接近理论上界')

## 3 · 温度 τ 的效应

τ 越小，softmax 越尖锐、越强调最难的负样本（与正对最像的那个）。
固定表示、扫 τ，看损失如何随 τ 变化。

In [ ]:
z = rng.standard_normal((6, 16))
z1 = z + 0.05*rng.standard_normal((6, 16))
z2 = z + 0.05*rng.standard_normal((6, 16))
print(f"{'tau':>6} {'InfoNCE':>10}")
losses = []
for tau in [0.05, 0.1, 0.5, 1.0]:
    L = info_nce(z1, z2, tau=tau)
    losses.append(L)
    print(f'{tau:>6.2f} {L:>10.4f}')
# 同一对(好)表示, 温度只改变损失的尺度/锐度, 但始终非负、有限
assert all(l >= 0 and np.isfinite(l) for l in losses), '各温度下损失应非负有限'
assert info_nce(z1, z2, 0.5) >= 0
print('✅ 温度效应：τ 调节损失的锐度，数值稳定(已做 max 减法)')

## 4 · MAE 掩码重建（toy）

随机掩掉一张图的大部分块，用可见块**线性回归**预测被掩块(toy 重建器)，损失只在被掩块算 MSE。
验证：掩码比例越高，可见信息越少、重建越难(MSE 越大)——这正是 MAE 要 75% 高掩码的原因。

In [ ]:
def random_mask(n_patches, mask_ratio, seed=0):
    '''返回被掩块的布尔索引(True=被掩)。'''
    r = np.random.default_rng(seed)
    n_mask = int(round(mask_ratio * n_patches))
    idx = r.permutation(n_patches)[:n_mask]
    mask = np.zeros(n_patches, dtype=bool); mask[idx] = True
    return mask

def mae_reconstruct_mse(patches, mask):
    '''toy MAE: 用可见块的均值预测被掩块(最简重建器), 返回被掩块上的 MSE。
       (真实 MAE 用 Transformer 解码器; 这里用均值演示"只在被掩块算损失"的机制。)'''
    vis = patches[~mask]; hid = patches[mask]
    if len(hid) == 0 or len(vis) == 0:
        return 0.0
    pred = vis.mean(axis=0, keepdims=True)            # 用可见块均值作预测
    return float(((hid - pred) ** 2).mean())          # 只在被掩块算 MSE

# 16 个 patch, 每个 4 维; 让 patch 之间有结构(相邻相似)
P = np.cumsum(rng.standard_normal((16, 4)) * 0.3, axis=0)   # 平滑变化的序列
mse_low = np.mean([mae_reconstruct_mse(P, random_mask(16, 0.25, s)) for s in range(20)])
mse_high = np.mean([mae_reconstruct_mse(P, random_mask(16, 0.75, s)) for s in range(20)])
print(f'掩码 25% 平均重建 MSE={mse_low:.3f}')
print(f'掩码 75% 平均重建 MSE={mse_high:.3f}')
assert mse_high > mse_low, '掩码比例越高, 可见信息越少, 重建越难(MSE 更大)'
# 损失只在被掩块算: 可见块不参与
mask = random_mask(16, 0.5, 0)
assert mae_reconstruct_mse(P, mask) >= 0
print('✅ MAE 机制正确：高掩码更难重建；损失只在被掩块上算')

## 5 · 表示坍缩诊断 + linear probe 评估

**坍缩**=所有表示挤到一点(方差→0)。诊断：看表示的方差/有效秩。
**linear probe**=冻结表示、只训线性分类器，用准确率衡量表示质量(线性可分性)。

In [ ]:
def representation_variance(Z):
    '''表示的平均特征方差; 坍缩时趋近 0。'''
    return float(Z.var(axis=0).mean())

# 健康表示 vs 坍缩表示
healthy = rng.standard_normal((50, 16))
collapsed = np.ones((50, 16)) * 0.5 + 1e-4*rng.standard_normal((50, 16))  # 几乎全相同
vh, vc = representation_variance(healthy), representation_variance(collapsed)
print(f'健康表示方差={vh:.3f}  坍缩表示方差={vc:.6f}')
assert vh > 0.1 and vc < 0.01, '坍缩表示的方差应趋近 0'

def linear_probe_acc(Ztr, ytr, Zte, yte, K=10, lr=0.5, epochs=200, l2=1e-3):
    '''冻结表示 Z, 只训线性分类器, 返回测试准确率。'''
    r = np.random.default_rng(0); d = Ztr.shape[1]
    W = 0.01*r.standard_normal((d, K)); b = np.zeros(K)
    Y = np.zeros((len(ytr), K)); Y[np.arange(len(ytr)), ytr] = 1
    for _ in range(epochs):
        S = Ztr @ W + b; S -= S.max(1, keepdims=True)
        P = np.exp(S); P /= P.sum(1, keepdims=True)
        dZ = (P - Y) / len(Ztr)
        W -= lr*(Ztr.T @ dZ + l2*W); b -= lr*dZ.sum(0)
    return float((np.argmax(Zte @ W + b, 1) == yte).mean())

# 好表示(类可分) vs 坍缩表示, probe 准确率应天差地别
y = rng.integers(0, 3, 90)
good = np.eye(3)[y] * 3 + 0.3*rng.standard_normal((90, 3))   # 按类分开
bad = np.ones((90, 3)) + 0.01*rng.standard_normal((90, 3))   # 坍缩
acc_good = linear_probe_acc(good[:60], y[:60], good[60:], y[60:], K=3)
acc_bad = linear_probe_acc(bad[:60], y[:60], bad[60:], y[60:], K=3)
print(f'好表示 linear probe acc={acc_good:.3f}  坍缩表示 acc={acc_bad:.3f}')
assert acc_good > 0.8 and acc_bad < 0.6, 'probe 应区分好表示与坍缩表示'
print('✅ 坍缩诊断(方差) + linear probe(线性可分性) 正确')

---
## ✏️ 练习 1：余弦相似度矩阵

实现 `cosine_sim_matrix(Z)`：先 L2 归一化每行，再返回 `Z_norm @ Z_norm.T`（对角线应≈1）。

In [ ]:
def cosine_sim_matrix(Z):
    # TODO: Zn = Z / (norm(Z, axis=1, keepdims) + 1e-12); return Zn @ Zn.T
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Z = rng.standard_normal((5, 8))
S = cosine_sim_matrix(Z)
assert S.shape == (5, 5)
assert np.allclose(np.diag(S), 1.0, atol=1e-6), '自身余弦相似度应为 1'
assert np.allclose(S, S.T), '相似度矩阵应对称'
assert (S <= 1.0 + 1e-6).all() and (S >= -1.0 - 1e-6).all(), '余弦∈[-1,1]'
print('✅ 练习 1 通过：余弦相似度矩阵正确')

## ✏️ 练习 2：InfoNCE 单个样本损失

实现 `nce_loss_one(sims, pos_idx, tau)`：给一行相似度 `sims`(已含与所有样本的相似度, 自身位置已设 -1e9)、正样本下标 `pos_idx`、温度 `tau`，返回该样本的 InfoNCE 损失 `-log softmax(sims/τ)[pos_idx]`。

In [ ]:
def nce_loss_one(sims, pos_idx, tau=0.5):
    # TODO: s = sims/tau; s -= s.max(); logp = s - log(sum(exp(s))); return -logp[pos_idx]
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
sims = np.array([-1e9, 0.9, 0.1, 0.2])   # 自身=-1e9, 正样本在 idx=1(最大)
L = nce_loss_one(sims, pos_idx=1, tau=0.5)
assert L >= 0, '损失非负'
# 正样本相似度最高 -> 损失应较小; 若正样本是低相似度的 idx=2, 损失应更大
L_bad = nce_loss_one(sims, pos_idx=2, tau=0.5)
assert L < L_bad, '正样本相似度越高, 损失越小'
print(f'✅ 练习 2 通过：InfoNCE 单样本损失正确 (好={L:.3f} < 差={L_bad:.3f})')

## ✏️ 练习 3：被掩块上的重建 MSE

实现 `masked_mse(pred, target, mask)`：只在 `mask==True` 的位置算 `(pred-target)²` 的均值（MAE 只在被掩块算损失）。

In [ ]:
def masked_mse(pred, target, mask):
    # TODO: 只在 mask 为 True 处算 MSE; return ((pred[mask]-target[mask])**2).mean()
    #   (若无被掩元素返回 0.0)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
pred = np.array([1.0, 2.0, 3.0, 4.0])
tgt  = np.array([1.0, 0.0, 3.0, 0.0])
mask = np.array([False, True, False, True])   # 只在 idx 1,3 算
m = masked_mse(pred, tgt, mask)
# 被掩处误差: (2-0)^2=4, (4-0)^2=16 -> 均值 10
assert abs(m - 10.0) < 1e-9, '只在被掩块算 MSE'
# 可见块(idx 0,2)即使有误差也不算
assert abs(masked_mse(np.array([9.,9.]), np.array([0.,0.]), np.array([False,False]))) < 1e-12
print('✅ 练习 3 通过：被掩块 MSE 正确(可见块不计入)')

## ✏️ 练习 4：linear probe 准确率

实现 `probe_accuracy(W, b, Z, y)`：给训练好的线性 probe 参数 `(W,b)` 和测试表示 `Z`、标签 `y`，返回 `argmax(ZW+b)==y` 的准确率。

In [ ]:
def probe_accuracy(W, b, Z, y):
    # TODO: pred = argmax(Z@W+b, axis=1); return (pred==y).mean()
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
Z = np.array([[2.,0.],[0.,2.],[2.,0.]])
W = np.eye(2); b = np.zeros(2)
y = np.array([0, 1, 0])                  # 与 argmax 一致
acc = probe_accuracy(W, b, Z, y)
assert abs(acc - 1.0) < 1e-9, '全对应 acc=1'
y_bad = np.array([1, 0, 1])
assert abs(probe_accuracy(W, b, Z, y_bad)) < 1e-9, '全错应 acc=0'
print('✅ 练习 4 通过：linear probe 准确率正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def cosine_sim_matrix(Z):
    Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    return Zn @ Zn.T

In [ ]:
# 练习 2 参考答案
def nce_loss_one(sims, pos_idx, tau=0.5):
    s = sims / tau
    s = s - s.max()
    logp = s - np.log(np.exp(s).sum())
    return float(-logp[pos_idx])

In [ ]:
# 练习 3 参考答案
def masked_mse(pred, target, mask):
    if mask.sum() == 0:
        return 0.0
    return float(((pred[mask] - target[mask]) ** 2).mean())

In [ ]:
# 练习 4 参考答案
def probe_accuracy(W, b, Z, y):
    pred = np.argmax(Z @ W + b, axis=1)
    return float((pred == y).mean())

---
## 🧪 真实数据胶囊：optdigits 上的表示评估

在真实 optdigits 上对比两种「表示」的 linear probe 准确率：① 原始像素 vs ② 一个简单的非线性随机特征(模拟学到的表示)。体会 linear probe 如何衡量表示的线性可分性——SSL 论文的标准评测。

In [ ]:
def load_digits_or_synth(n=400, seed=0):
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        return d.data[:n].astype(float)/16.0, d.target[:n].astype(int), 'real optdigits'
    except Exception:
        r = np.random.default_rng(seed)
        y = r.integers(0,10,n); X = np.zeros((n,64))
        for i in range(n):
            X[i, y[i]*6:(y[i]*6+6)] = 1.0
            X[i] += 0.1*r.standard_normal(64)
        return np.clip(X,0,1), y, 'synthetic fallback'

X, y, src = load_digits_or_synth(400)
print('source:', src, '| X', X.shape)
ntr = 280
Xtr, ytr, Xte, yte = X[:ntr], y[:ntr], X[ntr:], y[ntr:]
# ① 原始像素表示
acc_pixel = linear_probe_acc(Xtr, ytr, Xte, yte, K=10)
# ② 非线性随机特征(固定随机投影 + ReLU, 模拟一个未训练但非线性的表示)
r = np.random.default_rng(1); Wp = r.standard_normal((64, 128))
feat = lambda M: np.maximum(M @ Wp, 0)            # ReLU 随机特征
acc_feat = linear_probe_acc(feat(Xtr), ytr, feat(Xte), yte, K=10)
print(f'linear probe 准确率: 原始像素={acc_pixel:.3f}  随机ReLU特征={acc_feat:.3f}')
assert 0 <= acc_pixel <= 1 and 0 <= acc_feat <= 1
assert acc_pixel > 0.7, '原始像素 + 线性 probe 在 optdigits 上应可分'
print('✅ 真实 optdigits 表示评估跑通：linear probe 给出可比的表示质量数字')

**🧪 胶囊练习**：实现 `probe_quality(Xtr, ytr, Xte, yte)`：用 `linear_probe_acc` 返回原始表示的测试准确率，并断言它在 (0,1] 且明显高于随机(0.1)。

In [ ]:
def probe_quality(Xtr, ytr, Xte, yte):
    # TODO: return linear_probe_acc(Xtr, ytr, Xte, yte, K=10)
    raise NotImplementedError

In [ ]:
# 自测
q = probe_quality(Xtr, ytr, Xte, yte)
assert 0 < q <= 1 and q > 0.5, 'probe 准确率应明显高于随机'
print(f'✅ 胶囊练习通过：表示质量(linear probe acc)={q:.3f}')

In [ ]:
# 📖 胶囊参考答案
def probe_quality(Xtr, ytr, Xte, yte):
    return linear_probe_acc(Xtr, ytr, Xte, yte, K=10)

### 小结
- **对比学习**：正对(同图两视图)拉近、负对(不同图)推开；增强的选择决定学到什么不变性。
- **InfoNCE/NT-Xent**：= 在 2N−1 候选里认出正样本的交叉熵；**必须排除自身**、用余弦相似度、温度 τ 调锐度。
- **MAE**：掩掉大部分(75%)、只在被掩块算 MSE；图像冗余高故需高掩码比例。
- **表示坍缩**：全挤到一点(方差→0)；对比靠负样本、MAE 靠重建目标天然避免。
- **linear probe**：冻结表示训线性头，衡量线性可分性——SSL 的标准评测协议。

🎉 至此你已走完核心 CV 的低层(01) → 分类(02) → 检测(03) → 分割(04) → 自监督(05) 全谱系，并亲手用 numpy 写对了卷积、Sobel/Canny、HOG、IoU/NMS/mAP、Dice/mIoU、InfoNCE/MAE/linear probe。